# `win_probability_engine` -- MC-Dropout Uncertainty-Aware Predictions

**Ported near-verbatim from `project_hrishav/win_probability.py`.** Wraps a trained `IPLTransformer` and answers a richer question than a plain point-estimate model: not just "what's the win probability right now?", but "...and how *confident* is the model in that number?"

**Why this ports with almost no changes**: like `transformer_model.py`, this module only operates on already-built `(T, INPUT_DIM)` feature matrices -- it has no idea whether those came from `src/game_state.py` (this project) or hrishav's original Cricsheet-based pipeline. Only the import path and the `MC_SAMPLES` constant (previously `config.MC_SAMPLES`) needed adjusting.

## The core idea: Monte Carlo Dropout

During training, `Dropout` layers randomly zero out some activations on every forward pass -- this is a regularisation technique. At normal inference time, PyTorch automatically *disables* dropout (`model.eval()`), so predictions become deterministic.

**MC Dropout deliberately keeps dropout ACTIVE at inference time** (`model.train()`, even though we're not training) and runs the *same* input through the model many times. Because dropout randomly perturbs the network differently on each pass, the resulting predictions form a distribution rather than a single number:

```
mean = E[p(win)]           <- the best point estimate
std  = Std[p(win)]         <- how uncertain the model is
CI95 = mean +/- 1.96*std   <- a 95% confidence interval
```

This is a cheap, principled way to get uncertainty estimates out of an already-trained model, without needing a separate Bayesian architecture or an ensemble of models.

In [1]:
from typing import Optional

from dataclasses import dataclass

import numpy as np
import torch

from src.transformer_model import IPLTransformer

MC_SAMPLES = 50

## `WinProbResult` / `MatchWinProb` -- result containers

Plain `dataclass`es holding the full per-ball trajectory for one innings (`WinProbResult`: mean, std, CI bounds, projected score with its own mean/std, and human-readable over labels for plotting) or both innings of a full match (`MatchWinProb`: either innings can be `None` if that innings hasn't happened yet -- e.g. showing a live pre-2nd-innings prediction).

In [2]:
@dataclass
class WinProbResult:
    """Holds the full per-ball win-probability trajectory for one innings."""
    ball_indices: np.ndarray
    win_prob_mean: np.ndarray
    win_prob_std: np.ndarray
    ci_lower: np.ndarray
    ci_upper: np.ndarray
    score_mean: np.ndarray
    score_std: np.ndarray
    over_labels: list[str]

In [3]:
@dataclass
class MatchWinProb:
    """Win-probability results for both innings of a match."""
    innings1: Optional[WinProbResult]
    innings2: Optional[WinProbResult]
    batting_team1: str
    batting_team2: str

## `WinProbabilityEngine` -- the class doing the actual work

Constructed once around an already-trained `IPLTransformer` (moved to whichever device -- CPU/GPU -- inference should run on) plus a chosen number of MC samples (default 50 -- enough for a reasonably stable mean/std estimate without being prohibitively slow).

**`predict()`** is the core method: given one innings' feature matrix,
1. Adds a batch dimension (the model always expects `(B, T, D)`).
2. Explicitly calls `self.model.train()` **not** because we're    training, but specifically to keep dropout active -- the    comment in the code calls this out directly since it looks    like a mistake at first glance.
3. Runs `mc_samples` (50) independent forward passes inside    `torch.no_grad()` (no gradients needed, we're not    backpropagating -- this saves memory and compute), storing    each pass's win-probability and projected-score outputs.
4. Immediately calls `self.model.eval()` afterward, so the    engine leaves the model in normal inference mode for    anything else that might use it next -- this class never    permanently changes the model's mode.
5. Computes mean/std across the 50 samples at every ball    position, and derives the 95% CI (`mean +/- 1.96*std`,    clipped to the valid `[0,1]` probability range since the    normal-approximation CI could otherwise stray outside it).

**`predict_match()`** is a convenience wrapper calling `predict()` on both innings of a match (skipping cleanly if one hasn't happened yet).

**`update()`** is designed for a live/streaming use case: given the feature history *so far* in an ongoing innings, return just the latest ball's estimate -- useful for a dashboard that re-calls this after every new delivery.

In [4]:
class WinProbabilityEngine:
    """
    Parameters
    ----------
    model      : trained IPLTransformer (weights loaded, on correct device)
    mc_samples : number of stochastic forward passes
    device     : torch device string
    """

    def __init__(self, model: IPLTransformer, mc_samples: int = MC_SAMPLES, device: str = "cpu"):
        self.model = model.to(device)
        self.mc_samples = mc_samples
        self.device = device

    def predict(self, features: np.ndarray) -> WinProbResult:
        """
        Run MC Dropout inference on a single innings feature sequence.

        Parameters
        ----------
        features : (T, INPUT_DIM) float32 array, one row per delivery
        """
        T = features.shape[0]
        x = torch.tensor(features, dtype=torch.float32, device=self.device).unsqueeze(0)

        wp_samples = np.zeros((self.mc_samples, T), dtype=np.float32)
        score_samples = np.zeros((self.mc_samples, T), dtype=np.float32)

        self.model.train()  # keep dropout active -- this is the MC Dropout trick
        with torch.no_grad():
            for s in range(self.mc_samples):
                out = self.model(x)
                wp_samples[s] = out["win_prob"].squeeze().cpu().numpy()
                score_samples[s] = out["score_proj"].squeeze().cpu().numpy() * 250.0
        self.model.eval()

        mean_wp = wp_samples.mean(axis=0)
        std_wp = wp_samples.std(axis=0)

        return WinProbResult(
            ball_indices=np.arange(T),
            win_prob_mean=mean_wp,
            win_prob_std=std_wp,
            ci_lower=np.clip(mean_wp - 1.96 * std_wp, 0.0, 1.0),
            ci_upper=np.clip(mean_wp + 1.96 * std_wp, 0.0, 1.0),
            score_mean=score_samples.mean(axis=0),
            score_std=score_samples.std(axis=0),
            over_labels=_make_over_labels(T),
        )

    def predict_match(
        self,
        inn1_features: Optional[np.ndarray],
        inn2_features: Optional[np.ndarray],
        batting_team1: str = "",
        batting_team2: str = "",
    ) -> MatchWinProb:
        """Predict win-probability curves for both innings of a match."""
        return MatchWinProb(
            innings1=self.predict(inn1_features) if inn1_features is not None else None,
            innings2=self.predict(inn2_features) if inn2_features is not None else None,
            batting_team1=batting_team1,
            batting_team2=batting_team2,
        )

    def update(self, feature_history: np.ndarray) -> tuple[float, float, float]:
        """
        Given the feature history so far, return the LATEST win-probability
        estimate (mean, lower, upper). For a live-dashboard loop.
        """
        result = self.predict(feature_history)
        t = len(feature_history) - 1
        return float(result.win_prob_mean[t]), float(result.ci_lower[t]), float(result.ci_upper[t])

## `_make_over_labels()` and `summarise_uncertainty()` -- display helpers

`_make_over_labels` converts a raw ball index into the familiar cricket notation "over.ball" (e.g. `"12.4"` = the 4th ball of the 13th over), for use as x-axis labels on any plot built from a `WinProbResult`.

`summarise_uncertainty` prints a compact text table of the win probability, its 95% CI, and its standard deviation at regular checkpoints (every `every_n_overs` overs, plus always the final ball) -- a quick way to sanity-check a `WinProbResult` without needing a plotting library.

In [5]:
def _make_over_labels(T: int) -> list[str]:
    """Generate ball labels like '0.1', '0.2', ..., '19.6' for plotting."""
    labels = []
    legal_ball = 0
    for _ in range(T):
        over = legal_ball // 6
        ball_in = legal_ball % 6 + 1
        labels.append(f"{over}.{ball_in}")
        legal_ball += 1
    return labels

In [6]:
def summarise_uncertainty(result: WinProbResult, every_n_overs: int = 5) -> None:
    """Print a compact table of win-probability at key overs."""
    checkpoints = list(range(0, len(result.ball_indices), every_n_overs * 6))
    checkpoints.append(len(result.ball_indices) - 1)

    print(f"\n{'Over':>6}  {'P(win)':>8}  {'CI 95%':>18}  {'sigma':>6}")
    print("-" * 46)
    for t in checkpoints:
        over = t // 6
        print(
            f"{over:>6}  "
            f"{result.win_prob_mean[t]:>8.3f}  "
            f"[{result.ci_lower[t]:.3f}, {result.ci_upper[t]:.3f}]  "
            f"{result.win_prob_std[t]:>6.4f}"
        )